# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedahmed02/Flyrank-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# ============================================================
# ML-05 — Setup and Build March/April Dataset
# ============================================================

!pip -q install -U duckdb huggingface_hub pyarrow scikit-learn

import os
import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata

SEED = 42

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

os.environ["HF_TOKEN"] = HF_TOKEN

print("Environment ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 72.7 MB/s eta 0:00:00
Environment ready.


In [3]:
# ============================================================
# Build the March/April content-level dataset
# ============================================================

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HTTP,
    EXTRA_HTTP_HEADERS MAP {{
        'Authorization': 'Bearer {HF_TOKEN}'
    }}
);
""")

MARCH_URL = (
    "https://huggingface.co/datasets/"
    "FlyRank/internship-warehouse/resolve/main/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

APRIL_URL = (
    "https://huggingface.co/datasets/"
    "FlyRank/internship-warehouse/resolve/main/"
    "fact_content_daily_performance/month=2026-04/data_0.parquet"
)

dataset = con.execute(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_impressions) AS gsc_impressions,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_sum_position) AS DOUBLE)
                 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_avg_position,

        SUM(ga4_sessions) AS ga4_sessions,
        SUM(scroll_events) AS scroll_events

    FROM read_parquet('{MARCH_URL}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks

    FROM read_parquet('{APRIL_URL}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.*,
    a.april_clicks,

    CASE
        WHEN a.april_clicks > m.gsc_clicks THEN 1
        ELSE 0
    END AS label

FROM march m

INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
    AND m.content_hash_id = a.content_hash_id
""").fetchdf()

print("Rows:", len(dataset))
print("Columns:", list(dataset.columns))
print("Positive rate:", dataset["label"].mean())

display(dataset.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 158549
Columns: ['client_hash_id', 'content_hash_id', 'gsc_clicks', 'gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'scroll_events', 'april_clicks', 'label']
Positive rate: 0.17622943064919994


,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_sessions,scroll_events,april_clicks,label
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.0,6523.0,6.893301,1.0,0.0,8.0,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,0.0,453.0,3.214128,0.0,0.0,2.0,1
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,6.0,5630.0,6.535346,3.0,0.0,4.0,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,13.0,4944.0,7.435680,2.0,0.0,8.0,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,1.0,429.0,3.871795,2.0,1.0,0.0,0


## 1. Build the feature vector

The feature vector contains five March 2026 decision-time signals:

- GSC clicks
- GSC impressions
- GSC average position
- GA4 sessions
- Scroll events

April clicks and the target label are excluded from the feature vector because they represent the future outcome rather than information available at prediction time.

In [4]:
# ============================================================
# ML-05 — Section 1: Build the feature vector
# ============================================================

feature_cols = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events",
]

X = dataset[feature_cols].copy()
y = dataset["label"].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature columns:")
for col in feature_cols:
    print("-", col)

print("\nMissing values:")
display(X.isna().sum().to_frame("missing_count"))

# Leakage guard
assert "label" not in X.columns
assert "april_clicks" not in X.columns

print("\nLeakage guard passed.")

Feature matrix shape: (158549, 5)
Target shape: (158549,)

Feature columns:
- gsc_clicks
- gsc_impressions
- gsc_avg_position
- ga4_sessions
- scroll_events

Missing values:


,missing_count
gsc_clicks,0
gsc_impressions,0
gsc_avg_position,0
ga4_sessions,44696
scroll_events,44696



Leakage guard passed.


## 2. Feature notes (meaning, missing, categorical, available-when?)

All five features are numeric and represent information available during the March 2026 decision window.

- **GSC clicks:** Organic clicks received by the content item during March. No missing values. Available before the April outcome window.
- **GSC impressions:** Search impressions during March. No missing values. Available before the April outcome window.
- **GSC average position:** Impression-weighted average search position during March. No missing values. Available before the April outcome window.
- **GA4 sessions:** Sessions attributed to the content item during March. 44,696 observations are missing. Missing values are preserved and will be handled explicitly during modeling.
- **Scroll events:** Recorded scroll events during March. 44,696 observations are missing. Missing values are preserved and will be handled explicitly during modeling.

No categorical features are used in the model. The hashed client and content identifiers are kept only for grouping and joining and are not used as predictive features.

In [5]:
# ============================================================
# ML-05 — Section 2: Feature notes and availability checks
# ============================================================

feature_notes = pd.DataFrame({
    "feature": feature_cols,
    "dtype": [X[col].dtype for col in feature_cols],
    "missing_count": [X[col].isna().sum() for col in feature_cols],
    "missing_pct": [
        X[col].isna().mean() * 100
        for col in feature_cols
    ],
    "available_at_prediction": [True] * len(feature_cols)
})

display(feature_notes)

print("\nAll modeling features are numeric:")
print(all(pd.api.types.is_numeric_dtype(X[col]) for col in feature_cols))

print("\nClient ID used as feature:", "client_hash_id" in feature_cols)
print("Content ID used as feature:", "content_hash_id" in feature_cols)

assert all(pd.api.types.is_numeric_dtype(X[col]) for col in feature_cols)
assert "client_hash_id" not in feature_cols
assert "content_hash_id" not in feature_cols

print("\nFeature availability and identifier checks passed.")

,feature,dtype,missing_count,missing_pct,available_at_prediction
0,gsc_clicks,float64,0,0.000000,True
1,gsc_impressions,float64,0,0.000000,True
2,gsc_avg_position,float64,0,0.000000,True
3,ga4_sessions,float64,44696,28.190654,True
4,scroll_events,float64,44696,28.190654,True



All modeling features are numeric:
True

Client ID used as feature: False
Content ID used as feature: False

Feature availability and identifier checks passed.


## 3. The leakage hunt

The main leakage risks are future outcome fields, label-derived fields, and identifiers that could accidentally encode client-specific information.

April clicks are used only to construct the future label and are not included in the feature vector. The label itself is also excluded from the features.

Hashed client and content IDs are retained for joining and validation purposes only and are not predictive features.

No future-window feature is used as a March decision-time feature.

In [6]:
# ============================================================
# ML-05 — Section 3: Leakage hunt
# ============================================================

# Fields that must never enter the feature vector
forbidden_feature_cols = [
    "april_clicks",
    "label",
    "client_hash_id",
    "content_hash_id",
]

print("Checking forbidden fields...")

for col in forbidden_feature_cols:
    print(f"{col}: {'EXCLUDED' if col not in X.columns else 'FOUND'}")

# Explicit leakage assertions
assert "april_clicks" not in X.columns
assert "label" not in X.columns
assert "client_hash_id" not in X.columns
assert "content_hash_id" not in X.columns

# Check for obvious future/label-derived feature names
leakage_keywords = [
    "april",
    "label",
    "trend_direction",
    "trend_pct",
]

suspicious_features = [
    col for col in X.columns
    if any(keyword in col.lower() for keyword in leakage_keywords)
]

print("\nPotential future/label-derived features:", suspicious_features)

assert suspicious_features == []

print("\nLeakage check passed.")

Checking forbidden fields...
april_clicks: EXCLUDED
label: EXCLUDED
client_hash_id: EXCLUDED
content_hash_id: EXCLUDED

Potential future/label-derived features: []

Leakage check passed.


## 4. What I excluded and why

The following fields were deliberately excluded from the predictive feature vector:

- **April clicks:** Future-period information used only to define the outcome label. Including it would leak future information into the prediction.
- **Label:** The target variable itself. It cannot be used as an input feature.
- **Client hash ID:** A pseudonymous identifier used for grouping and joins, not a predictive signal.
- **Content hash ID:** A pseudonymous content identifier used for joins, not a meaningful predictive signal.
- **Trend-derived fields such as `trend_direction` and `trend_pct`:** These can be derived from outcome or future-period information and could introduce label leakage.
- **Private/client-identifying fields:** Excluded from the public-facing analysis to maintain data safety and prevent disclosure of business information.

The final feature vector contains only five March decision-time numeric signals: GSC clicks, GSC impressions, GSC average position, GA4 sessions, and scroll events.

In [7]:
# ============================================================
# ML-05 — Section 4: Excluded fields and final feature check
# ============================================================

excluded_fields = {
    "april_clicks": "Future-period outcome information; would cause leakage.",
    "label": "Target variable; cannot be used as an input feature.",
    "client_hash_id": "Pseudonymous identifier used only for grouping and joins.",
    "content_hash_id": "Pseudonymous identifier used only for joins.",
    "trend_direction": "Outcome-derived field; potential label leakage.",
    "trend_pct": "Outcome-derived field; potential label leakage.",
}

excluded_table = pd.DataFrame(
    list(excluded_fields.items()),
    columns=["excluded_field", "reason"]
)

display(excluded_table)

print("Final feature vector:")
print(feature_cols)

print("\nFinal feature count:", len(feature_cols))
print("Rows:", len(X))

assert len(feature_cols) == 5
assert set(feature_cols) == {
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events",
}

print("\nFinal feature check passed.")

,excluded_field,reason
0,april_clicks,Future-period outcome information; would cause...
1,label,Target variable; cannot be used as an input fe...
2,client_hash_id,Pseudonymous identifier used only for grouping...
3,content_hash_id,Pseudonymous identifier used only for joins.
4,trend_direction,Outcome-derived field; potential label leakage.
5,trend_pct,Outcome-derived field; potential label leakage.


Final feature vector:
['gsc_clicks', 'gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'scroll_events']

Final feature count: 5
Rows: 158549

Final feature check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.